In [ ]:
%pip install -q kagglehub gradio

import ast
import pickle
import re
from difflib import get_close_matches
from html import escape
from pathlib import Path

import gradio as gr
import kagglehub
import numpy as np
import pandas as pd
from IPython.display import FileLink
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ps = PorterStemmer()


In [ ]:
path = kagglehub.dataset_download("artermiloff/steam-games-dataset")

print("Downloaded to:", path)

Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Downloaded to: /kaggle/input/steam-games-dataset


In [ ]:
csv_files = list(Path(path).rglob("*.csv"))

for file in csv_files:
    print(file.name)

games_may2024_cleaned.csv
games_may2024_full.csv
games_march2025_full.csv
games_march2025_cleaned.csv


In [ ]:
csv_path = next(
    (file for file in csv_files
     if file.name == "games_march2025_cleaned.csv"),
    None
)

if csv_path is None:
    print("File name changed. Check the filenames printed above.")
else:
    df = pd.read_csv(csv_path, nrows=20000)
    display(df.head())

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,3.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,682,2434,306,80,89916,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     20000 non-null  int64  
 1   name                      20000 non-null  object 
 2   release_date              20000 non-null  object 
 3   required_age              20000 non-null  int64  
 4   price                     20000 non-null  float64
 5   dlc_count                 20000 non-null  int64  
 6   detailed_description      19930 non-null  object 
 7   about_the_game            19916 non-null  object 
 8   short_description         19963 non-null  object 
 9   reviews                   5245 non-null   object 
 10  header_image              20000 non-null  object 
 11  website                   12825 non-null  object 
 12  support_url               12009 non-null  object 
 13  support_email             16248 non-null  object 
 14  window

In [ ]:
df.isnull().sum()

,0
appid,0
name,0
release_date,0
required_age,0
price,0
dlc_count,0
detailed_description,70
about_the_game,84
short_description,37
reviews,14755


In [ ]:
df = df[['appid', 'name', 'price', 'detailed_description', 'about_the_game', 'short_description', 'header_image', 'developers', 'publishers', 'genres']]

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

(19916, 10)

In [ ]:
print(df['price'].value_counts().sort_index())


price
0.00     4134
0.49      112
0.50        1
0.54        2
0.55       23
         ... 
59.99     198
64.99       1
69.99      35
79.99       4
89.99       1
Name: count, Length: 325, dtype: int64


In [ ]:
df['price']  = df['price'].astype('string')

In [ ]:
df['detailed_description'] = df['detailed_description'].apply(lambda x: x.split())
df['about_the_game'] = df['about_the_game'].apply(lambda x: x.split())
df['short_description'] = df['short_description'].apply(lambda x: x.split())
df['price'] = df['price'].apply(lambda x: [x])

In [ ]:
df[["detailed_description","about_the_game","short_description","price"]].head()

,detailed_description,about_the_game,short_description,price
0,"[For, over, two, decades,, Counter-Strike, has...","[For, over, two, decades,, Counter-Strike, has...","[For, over, two, decades,, Counter-Strike, has...",[0.0]
1,"[LAND,, LOOT,, SURVIVE!, Play, PUBG:, BATTLEGR...","[LAND,, LOOT,, SURVIVE!, Play, PUBG:, BATTLEGR...","[Play, PUBG:, BATTLEGROUNDS, for, free., Land,...",[0.0]
2,"[The, most-played, game, on, Steam., Every, da...","[The, most-played, game, on, Steam., Every, da...","[Every, day,, millions, of, players, worldwide...",[0.0]
3,"[When, a, young, street, hustler,, a, retired,...","[When, a, young, street, hustler,, a, retired,...","[Grand, Theft, Auto, V, for, PC, offers, playe...",[0.0]
4,"[Edition, Comparison, Ultimate, Edition, The, ...","[“One, of, the, best, first-person, shooters, ...","[Tom, Clancy's, Rainbow, Six®, Siege, is, an, ...",[3.99]


In [ ]:
df['genres'] = df['genres'].apply(ast.literal_eval)
df['developers'] = df['developers'].apply(ast.literal_eval)
df['publishers'] = df['publishers'].apply(ast.literal_eval)

In [ ]:
df[['genres','developers','publishers']].head()

,genres,developers,publishers
0,"[Action, Free To Play]",[Valve],[Valve]
1,"[Action, Adventure, Massively Multiplayer, Fre...",[PUBG Corporation],"[KRAFTON, Inc.]"
2,"[Action, Strategy, Free To Play]",[Valve],[Valve]
3,"[Action, Adventure]",[Rockstar North],[Rockstar Games]
4,[Action],[Ubisoft Montreal],[Ubisoft]


In [ ]:
df['detailed_description'] = df['detailed_description'].apply(lambda x:[i.replace(" ","") for i in x])
df['about_the_game'] = df['about_the_game'].apply(lambda x:[i.replace(" ","") for i in x])
df['short_description'] = df['short_description'].apply(lambda x:[i.replace(" ","") for i in x])
df['genres'] = df['genres'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
df['tags'] = df['price'] + df['detailed_description'] + df['about_the_game'] + df['short_description'] + df['genres']

In [ ]:
df['tags'].head()

,tags
0,"[0.0, For, over, two, decades,, Counter-Strike..."
1,"[0.0, LAND,, LOOT,, SURVIVE!, Play, PUBG:, BAT..."
2,"[0.0, The, most-played, game, on, Steam., Ever..."
3,"[0.0, When, a, young, street, hustler,, a, ret..."
4,"[3.99, Edition, Comparison, Ultimate, Edition,..."


In [ ]:
df=df.drop(df[df['name']=="Furry Arena [18+]"].index)#include any game that is missing with recommendation here
df[df['name']=="Furry Arena [18+]"]

,appid,name,price,detailed_description,about_the_game,short_description,header_image,developers,publishers,genres,tags


In [ ]:
new_df = df[['appid','name','tags']]


In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


/tmp/ipykernel_18514/224074193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [ ]:
new_df["tags"][0]

'0.0 For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools. In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features: All-new CS Ratings with the updated Premier mode Global and Regional leaderboards Upgraded and overhauled maps Game-changing dynamic smoke grenades Tick-rate-independent gameplay Redesigned visual effects and audio All items from CS:GO moving forward to CS2 For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players 

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


/tmp/ipykernel_18514/1889842218.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)


/tmp/ipykernel_18514/1132361295.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [ ]:
cv = CountVectorizer(stop_words = 'english',max_features = 5000)

In [ ]:
vectorrs = cv.fit_transform(new_df['tags'])

In [ ]:
cv.get_feature_names_out()

array(['00', '000', '10', ..., 'zones', 'zoo', 'zoom'], dtype=object)

In [ ]:
vectorrs

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2090997 stored elements and shape (19915, 5000)>

In [ ]:
similiarity = cosine_similarity(vectorrs)


In [ ]:
similiarity

array([[1.        , 0.04411996, 0.13122524, ..., 0.04268701, 0.04417135,
        0.10938699],
       [0.04411996, 1.        , 0.16057638, ..., 0.0477703 , 0.03259606,
        0.06089427],
       [0.13122524, 0.16057638, 1.        , ..., 0.20869735, 0.15629777,
        0.10554802],
       ...,
       [0.04268701, 0.0477703 , 0.20869735, ..., 1.        , 0.19528927,
        0.1262425 ],
       [0.04417135, 0.03259606, 0.15629777, ..., 0.19528927, 1.        ,
        0.12463997],
       [0.10938699, 0.06089427, 0.10554802, ..., 0.1262425 , 0.12463997,
        1.        ]])

In [ ]:
def recommend(game):
  game_id = new_df[new_df['name'] == game].index[0]
  distances = similiarity[game_id]
  game_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

  for i in game_list:
    print(new_df.iloc[i[0]]['name'])

In [ ]:
recommend('Terraria')

Creativerse
RPG World - Action RPG Maker
Garage: Bad Dream Adventure
Omega Crafter
Teravit


In [ ]:
games_with_posters = df[['appid', 'name', 'header_image']]

In [ ]:
games = new_df.reset_index(drop=True)
scores_matrix = similiarity

match_column = (
    "appid"
    if "appid" in games.columns and "appid" in df.columns
    else "name"
)
image_lookup = (
    df.dropna(subset=[match_column, "header_image"])
    .drop_duplicates(subset=[match_column])
    .set_index(match_column)["header_image"]
    .to_dict()
)

choices = sorted(
    [
        (str(row["name"]), int(index))
        for index, row in games.iterrows()
    ],
    key=lambda item: item[0].lower()
)

In [ ]:
def recommend_cards(game_index):
    if game_index is None:
        return "<p>Please select a game first.</p>"

    game_index = int(game_index)
    scores = scores_matrix[game_index]

    if hasattr(scores, "toarray"):
        scores = scores.toarray()

    scores = np.asarray(scores).ravel()

    recommended = [
        int(i)
        for i in np.argsort(-scores)
        if i != game_index and np.isfinite(scores[i])
    ][:5]

    cards = []

    for index in recommended:
        game = games.iloc[index]
        name = escape(str(game["name"]))
        image_url = image_lookup.get(game[match_column])

        if isinstance(image_url, str) and image_url.startswith(
            ("https://", "http://")
        ):
            picture = f"""
                <img
                    src="{escape(image_url, quote=True)}"
                    alt="{name}"
                    loading="lazy"
                    style="
                        display: block;
                        width: 100%;
                        height: 100%;
                        object-fit: contain;
                    "
                >
            """
        else:
            picture = "<span>Image unavailable</span>"

        cards.append(f"""
            <div style="
                background: #182235;
                border: 1px solid #334155;
                border-radius: 14px;
                overflow: hidden;
            ">
                <div style="
                    padding: 16px;
                    min-height: 76px;
                    box-sizing: border-box;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    text-align: center;
                    overflow-wrap: anywhere;
                    color: white;
                    font-size: 17px;
                    font-weight: 700;
                ">
                    {name}
                </div>

                <div style="
                    width: 100%;
                    aspect-ratio: 460 / 215;
                    background: #0b1120;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    color: #94a3b8;
                ">
                    {picture}
                </div>
            </div>
        """)

    return f"""
        <div style="
            display: grid;
            grid-template-columns:
                repeat(auto-fit, minmax(min(100%, 260px), 1fr));
            gap: 20px;
            max-width: 1100px;
            margin: 20px auto;
        ">
            {''.join(cards)}
        </div>
    """

In [ ]:
# Read categories from the dataset.
category_column = next(
    (column for column in ["genres", "genre", "categories"]
     if column in df.columns),
    None
)
games

def read_categories(value):
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]

    if not isinstance(value, str) or not value.strip():
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]
    except (ValueError, SyntaxError):
        pass

    return [
        item.strip(" '\"[]")
        for item in re.split(r"[,;|]", value)
        if item.strip(" '\"[]")
    ]


search_games = games.copy()
search_games["_search_name"] = (
    search_games["name"].fillna("").astype(str).str.casefold()
)

if category_column:
  category_lookup = (
    df.dropna(subset=[match_column])
    .drop_duplicates(subset=[match_column])
    .set_index(match_column)[category_column]
)


  search_games["_genres"] = [
        read_categories(category_lookup.get(key))
        for key in search_games[match_column]
    ]
else:
    search_games["_genres"] = [[] for _ in range(len(search_games))]

all_categories = sorted({
    category
    for categories in search_games["_genres"]
    for category in categories
})

def search_titles(query, category):
    available = search_games

    if category != "All categories":
        available = available[
            available["_genres"].apply(
                lambda categories: category in categories
            )
        ]

    query = (query or "").strip().casefold()
    suggested = False

    if query:
        matches = available[
            available["_search_name"].str.contains(query, regex=False)
        ].copy()

        # Show exact matches first, then names starting with your search.
        if not matches.empty:
            matches["_rank"] = matches["_search_name"].apply(
                lambda name: (
                    0 if name == query
                    else 1 if name.startswith(query)
                    else 2
                )
            )
            matches = matches.sort_values(["_rank", "_search_name"])

        # Suggest similar names if the spelling is slightly wrong.
        else:
            close_names = get_close_matches(
                query,
                available["_search_name"].unique().tolist(),
                n=10,
                cutoff=0.55
            )
            matches = available[
                available["_search_name"].isin(close_names)
            ].copy()

            if close_names:
                order = {name: i for i, name in enumerate(close_names)}
                matches["_rank"] = matches["_search_name"].map(order)
                matches = matches.sort_values("_rank")
                suggested = True
    else:
        matches = available.sort_values("_search_name")

    choices = [
        (str(row["name"]), int(index))
        for index, row in matches.head(50).iterrows()
    ]

    if not choices:
        message = "No games found. Try a shorter name or another category."
    elif suggested:
        message = "No exact search results. Did you mean one of these games?"
    else:
        message = (
            f"{len(matches):,} games found. "
            f"Showing {len(choices)}—select a game below."
        )

    return (
        gr.Dropdown(choices=choices, value=None),
        message,
        ""
    )


initial_choices, initial_message, _ = search_titles("", "All categories")



In [ ]:
def games_in_category(category):
    available = search_games

    if category != "All games":
        available = available[
            available["_genres"].apply(
                lambda genres: category in genres
            )
        ]

    return sorted(
        [
            (str(row["name"]), int(index))
            for index, row in available.iterrows()
        ],
        key=lambda item: item[0].lower()
    )

def change_category(category):
    return (
        gr.Dropdown(
            choices=games_in_category(category),
            value=None
        ),
        ""
    )

In [ ]:
from transformers import pipeline

LLM=pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",
    device="cuda"
    )

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
def message(user_msg):
  msg=[
      {
      "role":"system",
      "content":"""You turn the user's input into game preferences and game genres.
examples: user asks for a relaxing game where they can build, you will return preferences and genres such as relaxing, low difficulty, short sessions, casual gameplay, building, and sandbox,
if the user asks for a competitive shooter, you will return preferences and genres such as first-person shooter, competitive multiplayer, fast-paced action, quick reflexes, tactical
Return a small range of preferences and genres that closely match what the user is looking for.
Answer with the preferences and genres only, using the same format as in the example.
      """
      },
      {
      "role":"user",
      "content":user_msg
       }
       ]
  return msg

In [ ]:
user_msg="I want a scary survival game"
reply=LLM(message(user_msg))
reply[0]["generated_text"][2]["content"]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'scary, survival, horror, intense action, exploration, stealth, dark atmosphere'

In [ ]:
def recommend_by_user_message(user_msg):
  prefrence=LLM(message(user_msg))[0]["generated_text"][2]["content"].replace("\n","").replace("-","").replace(","," ")
  original_vector = cv.transform([stem(user_msg)])
  preference_vector = cv.transform([stem(prefrence)])

  query_vector =  original_vector + preference_vector

  scores=cosine_similarity(
      query_vector,
      vectorrs
  ).ravel()

  top_games=np.argsort(scores)[::-1][:5]

  return [new_df.iloc[i]["name"] for i in top_games]

In [ ]:
recommend_by_user_message(user_msg)


['Blame Him', 'Pitty Meaty', 'Uncanny Valley', 'Outlast', 'The Long Dark']

In [ ]:

def recommend_by_user_message_cards(user_msg):
    if not user_msg or not user_msg.strip():
        return "<p>Please describe the game you want.</p>"

    recommended_names = recommend_by_user_message(user_msg.strip())
    cards = []

    for game_name in recommended_names:
        matches = games[games["name"].astype(str) == str(game_name)]

        if matches.empty:
            continue

        game = matches.iloc[0]
        name = escape(str(game["name"]))
        image_url = image_lookup.get(game[match_column])

        if isinstance(image_url, str) and image_url.startswith(
            ("https://", "http://")
        ):
            picture = f"""
                <img
                    src="{escape(image_url, quote=True)}"
                    alt="{name}"
                    loading="lazy"
                    style="width:100%;height:100%;object-fit:contain;"
                >
            """
        else:
            picture = "<span>Image unavailable</span>"

        cards.append(f"""
            <div style="
                background:#182235;
                border:1px solid #334155;
                border-radius:14px;
                overflow:hidden;
            ">
                <div style="
                    padding:16px;
                    min-height:76px;
                    display:flex;
                    align-items:center;
                    justify-content:center;
                    text-align:center;
                    overflow-wrap:anywhere;
                    color:white;
                    font-size:17px;
                    font-weight:700;
                ">
                    {name}
                </div>

                <div style="
                    aspect-ratio:460/215;
                    background:#0b1120;
                    display:flex;
                    align-items:center;
                    justify-content:center;
                    color:#94a3b8;
                ">
                    {picture}
                </div>
            </div>
        """)

    if not cards:
        return "<p>No matching games or images were found.</p>"

    return f"""
        <div style="
            display:grid;
            grid-template-columns:repeat(
                auto-fit,minmax(min(100%,260px),1fr)
            );
            gap:20px;
            max-width:1100px;
            margin:20px auto;
        ">
            {''.join(cards)}
        </div>
    """


In [ ]:

with gr.Blocks() as search_app:
    gr.Markdown("# 🎮 Discover Your Next Game")
    gr.Markdown("Choose how you want to discover games.")

    # The switch controls the visibility of the entire interface.
    recommendation_mode = gr.Radio(
        choices=[
            "Similar games",
            "Recommend by description"
        ],
        value="Similar games",
        label="Recommendation method",
        interactive=True
    )

    # -----------------------------------
    # MODE 1: Existing similar-games UI
    # -----------------------------------
    with gr.Group(visible=True) as similar_panel:
        gr.Markdown("## 🔎 Find Similar Games")

        with gr.Row():
            search_box = gr.Textbox(
                label="Search games",
                placeholder="Try GTA, FIFA, Minecraft...",
                scale=3
            )

            category_filter = gr.Dropdown(
                choices=["All categories"] + all_categories,
                value="All categories",
                label="Category",
                interactive=True,
                scale=1
            )

        search_status = gr.Markdown(initial_message)

        game_picker = gr.Dropdown(
            choices=initial_choices.choices,
            value=None,
            label="Search results — select a game",
            filterable=True,
            interactive=True
        )

        similar_button = gr.Button(
            "✨ Find Similar Games",
            variant="primary"
        )

        similar_cards = gr.HTML()

        # Preserve your existing search behavior.
        search_box.input(
            fn=search_titles,
            inputs=[search_box, category_filter],
            outputs=[game_picker, search_status, similar_cards]
        )

        category_filter.change(
            fn=search_titles,
            inputs=[search_box, category_filter],
            outputs=[game_picker, search_status, similar_cards]
        )

        similar_button.click(
            fn=recommend_cards,
            inputs=[game_picker],
            outputs=[similar_cards]
        )

    # -----------------------------------
    # MODE 2: Description-based UI
    # -----------------------------------
    with gr.Group(visible=False) as description_panel:
        gr.Markdown("## 💬 Describe Your Ideal Game")

        user_message_box = gr.Textbox(
            label="What kind of game are you looking for?",
            placeholder=(
                "Example: I want a relaxing open-world game "
                "where I can explore and build things."
            ),
            lines=3
        )

        description_button = gr.Button(
            "✨ Recommend Games",
            variant="primary"
        )

        description_cards = gr.HTML()

        description_button.click(
            fn=recommend_by_user_message_cards,
            inputs=[user_message_box],
            outputs=[description_cards]
        )

    # Switch the entire panel, not just the recommendation method.
    recommendation_mode.change(
        fn=lambda mode: (
            gr.update(visible=(mode == "Similar games")),
            gr.update(visible=(mode == "Recommend by description"))
        ),
        inputs=[recommendation_mode],
        outputs=[similar_panel, description_panel]
    )

search_app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b15302fb339e415d50.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
